# PySpark Examples with emp and dept Data (Microsoft Fabric Focus)

This notebook is a runnable conversion of the `PySpark_07_more used functions.md` notes. It contains markdown explanation cells and runnable PySpark code cells for common operations using the `emp` and `dept` datasets in a Fabric Lakehouse environment.

## Prerequisites
- A running Spark session in your Fabric notebook or Synapse environment.
- Files placed in the lakehouse under `/lakehouse/default/Files` or adjust paths accordingly.
- Delta support enabled (Fabric Lakehouse).
- Useful helpers: `display(df)` in notebooks to preview small results.

## 1. Reading CSV Files

In [ ]:
print("hi")

In [ ]:
# Recommended: include schema inference (or provide an explicit schema)
emp_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/lakehouse/default/Files/emp.csv")
dept_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/lakehouse/default/Files/dept.csv")

# Quick preview (not for very large files)
display(emp_df.limit(5))
display(dept_df.limit(5))

## 2. Casting and Filtering Rows

In [ ]:
from pyspark.sql.functions import col

# Ensure `sal` is numeric then filter employees with salary greater than 2000
emp_df = emp_df.withColumn("sal", col("sal").cast("double"))
emp_df_filtered = emp_df.filter(col("sal") > 2000)
display(emp_df_filtered.limit(10))

## 3. Column Calculations

In [ ]:
# Add bonus column (10% of salary)
emp_df_with_bonus = emp_df.withColumn("bonus", col("sal") * 0.1)
display(emp_df_with_bonus.select("ename", "sal", "bonus").limit(10))

## 4. Reading / Combining Parquet Files (avoid Python loops when possible)

In [ ]:
# Example: read multiple parquet files directly with Spark
paths = ["/lakehouse/default/Files/data1.parquet", "/lakehouse/default/Files/data2.parquet"]
# If these files exist, Spark will read them into one DataFrame
# multi_df = spark.read.parquet(*paths)

# If you must download remote files first, do so outside Spark and then read the local folder (example omitted for brevity).

## 5. Joins

In [ ]:
# Inner join on deptno
emp_dept_df = emp_df.join(dept_df, on="deptno", how="inner")
display(emp_dept_df.limit(10))

## 6. Broadcast Joins (Performance Optimization)

In [ ]:
from pyspark.sql.functions import broadcast

# Broadcast smaller dept table (use when dept is small, e.g., < 10-50 MB)
emp_dept_broadcast = emp_df.join(broadcast(dept_df), on="deptno", how="inner")
emp_dept_broadcast.explain(True)
display(emp_dept_broadcast.limit(10))

## 7. Grouping and Aggregations

In [ ]:
# Average salary by department
avg_salary_df = emp_df.groupBy("deptno").agg({"sal": "avg"})
display(avg_salary_df)


## 8. Window Functions

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

window_spec = Window.partitionBy("deptno").orderBy(col("sal").desc())
ranked_df = emp_df.withColumn("salary_rank", rank().over(window_spec))
display(ranked_df.limit(20))

## 9. Writing to Delta Table

In [ ]:
# Option A: save to a lakehouse path
# emp_dept_df.write.format("delta").mode("overwrite").save("/lakehouse/default/Tables/emp_dept_delta")

# Option B: save as a managed table (catalog) if you want it queryable via SQL
# emp_dept_df.write.mode("overwrite").saveAsTable("default.emp_dept_delta")

## 10. Caching and Persistence

In [ ]:
from pyspark import StorageLevel

# Cache frequently used DataFrame
emp_df.cache()
# Or persist with storage level
emp_df.persist(StorageLevel.MEMORY_AND_DISK)
# When done, free cache
# emp_df.unpersist()

## 11. Reading/Writing OneLake and file partitioning tips

In [ ]:
# Repartition vs coalesce examples
# Repartition before a big join to improve parallelism

,
deptno")  # adjust number of partitions to cluster size
result = emp_repart.join(dept_df, on="deptno", how="inner")

# Coalesce before writing to reduce number of output files
# result_final = result.coalesce(4)
# result_final.write.mode("overwrite").format("delta").save("/lakehouse/default/Files/emp_dept_joined")


: 
,
: {},
: [
,